# 05 · 标准推理流程(★★★★★)

加载权重 → fp16/eval/inference_mode → 读输入(72通道)→ 前向 → 反归一化 → 存 69 通道 npy。
**★计时区只包 `model(x)`;数据读取/保存不计入;输入72出69;保存前反归一化。**

In [ ]:
# ============ 公共设置(每个 notebook 先跑这一格)============
import os, sys, json, glob, math, time, numpy as np, torch, torch.nn.functional as F
import warnings; warnings.filterwarnings("ignore")

BASE = "/public/home/xdzs2026_c296"          # ★你的主目录,若不同改这里
BASELINE = f"{BASE}/xiandao2026-AI4S/pangu_weather"   # 官方 baseline(含 maxvit3d_student.py, conf, data)
CKPT = f"{BASELINE}/data/checkpoints/model_bak.pth"   # 教师权重
TRAIN_DATA = f"{BASE}/era5_real"              # 训练数据(13年)
VAL_DATA   = f"{BASE}/era5_testc"             # 验证/测试数据(2000年)
WORK = f"{BASE}/_learn_work"                  # 本教程的工作目录(存中间文件)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, BASELINE)                  # 为了 import maxvit3d_student
print("torch", torch.__version__, "| DCU 可用:", torch.cuda.is_available())


In [ ]:
# 造一个指向验证数据的 conf/config.yaml(官方 config 的 data_dir 指向 /work2 拿不到,改指 VAL_DATA)
os.makedirs(f"{WORK}/conf", exist_ok=True)
src = open(f"{BASELINE}/conf/config.yaml", encoding="utf-8").read()
src = src.replace("/work2/share/sugonhpcapp01/ERA5/old-data", VAL_DATA)
open(f"{WORK}/conf/config.yaml", "w", encoding="utf-8").write(src)
print("已写", f"{WORK}/conf/config.yaml", "-> data_dir =", VAL_DATA)


## 1) 载配置 + 建模型 + 载权重 + fp16/eval
(这里用教师 Pangu 演示完整推理;换学生只需 build 学生并 load 学生权重)

In [ ]:
from onescience.utils.YParams import YParams
from onescience.models.pangu import Pangu
cfg = YParams(f"{WORK}/conf/config.yaml", "model")
cfg_d = YParams(f"{WORK}/conf/config.yaml", "datapipe")
dev = 0
model = Pangu(img_size=tuple(cfg_d.dataset.img_size), patch_size=cfg.patch_size,
              embed_dim=cfg.embed_dim, num_heads=cfg.num_heads, window_size=cfg.window_size).to(dev)
model.load_state_dict(torch.load(CKPT, map_location=f"cuda:{dev}", weights_only=False)["model_state_dict"])
model = model.half().eval()          # ★fp16(V/U 双降,精度几乎不掉)
print("模型就绪(fp16)")

## 2) 归一化参数 + 静态场(half 常驻)

In [ ]:
ch = list(cfg_d.dataset.channels)
meta = json.load(open(f"{cfg_d.dataset.data_dir}/metadata.json"))["variables"]
sel = [meta.index(v) for v in ch]
mu = np.load(f"{cfg_d.dataset.data_dir}/stats/global_means.npy")[:, sel]   # (1,69,1,1)
sdv= np.load(f"{cfg_d.dataset.data_dir}/stats/global_stds.npy")[:, sel]
sdd = cfg_d.dataset.static_dir
land=np.load(f"{sdd}/land_mask.npy").astype(np.float32); soil=np.load(f"{sdd}/soil_type.npy").astype(np.float32)
topo=np.load(f"{sdd}/topography.npy").astype(np.float32); topo=(topo-topo.mean())/(topo.std()+1e-6)
surface_mask = torch.tensor(np.stack([land,soil,topo],0)).unsqueeze(0).half().to(dev)  # 静态场 half 常驻

## 3) 数据管线 + 推理循环(★计时只包 model(x))

In [ ]:
from onescience.datapipes.climate import ERA5Datapipe
os.makedirs(f"{WORK}/result/output", exist_ok=True)
dl = ERA5Datapipe(params=cfg_d, distributed=False).test_dataloader()
time_list = []
with torch.inference_mode():                       # ★比 no_grad 更省
    for i, data in enumerate(dl):
        invar = data[0]; filename = data[4][-1][0]          # 已归一 69 通道
        isf = invar[:, :4].half().to(dev); iua = invar[:, 4:].half().to(dev)
        x = torch.cat([isf, surface_mask, iua], dim=1)      # 72 通道(4面+3静态+65高空)
        torch.cuda.synchronize()
        # ---- AI4S 计时区:只此一句 ----
        t0 = time.perf_counter()
        out_s, out_u = model(x)
        torch.cuda.synchronize(); time_list.append(time.perf_counter()-t0)
        # -----------------------------
        out_u = out_u.reshape(1, 65, 721, 1440)
        pred = torch.cat([out_s.float(), out_u.float()], 1)[0].cpu().numpy()
        pred = pred * sdv.reshape(69,1,1) + mu.reshape(69,1,1)    # ★反归一化 -> 绝对场
        np.save(f"{WORK}/result/output/{filename}.npy", pred)
        if i >= 4: break        # 演示存 5 个;实战跑全部
json.dump(time_list, open(f"{WORK}/result/time_record.json","w"))
print("已存", len(time_list), "个 npy | 前向中位", round(float(np.median(time_list))*1000,1), "ms")

### ✅ 要点:`model.half().eval()`+`inference_mode`;输入拼 72;计时**只 synchronize 包 model(x)**;输出反归一化成绝对场存 npy。